# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset via the `mlcroissant` library, with a focus on programmatically referencing record sets, fields, and columns by their `@id` identifiers throughout.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure mlcroissant is installed (uncomment if running in a new environment)
!pip install mlcroissant

## 1. Data Loading
Load metadata and initialize the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print a summary (demonstrates accessing top-level metadata properties as attributes)
print(f"\nDataset: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")


## 2. Data Overview
List the available record sets and their respective fields as specified by `@id`.

In [ ]:
# Retrieve all record sets using their @id; demonstrate overview of record sets, fields, and columns
# Note: Must use .via_id and avoid treating metadata as subscriptable or iterable
print("Available record sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in fields:
            # Field can be an @id or dict
            if isinstance(field, dict) and '@id' in field:
                field_id = field['@id']
            else:
                field_id = field
            print(f"    - Field @id: {field_id}")


## 3. Data Extraction
Extract data from one or more record sets using their `@id`s, and load them as pandas DataFrames for further analysis.

**Note:** The `@id` of the main data record set can be found in the overview section above.

In [ ]:
# Create a mapping of record set @ids from the previous cell
# For illustration, we'll extract all top-level record sets
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

# Load each record set into a DataFrame key'd by its @id
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(df.head(), "\n")
    else:
        print(f"No records found for {record_set_id}\n")

# Example: Select the first record set for further steps
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nProceeding to analysis for record set: {main_record_set_id}")
else:
    main_record_set_id = None


## 4. Exploratory Data Analysis (EDA)
Filter, normalize, and group data based on numeric and grouping fields.
All field references use `@id` identifiers as described in the documentation and previous steps.

In [ ]:
import numpy as np

# Pick a record set and find its numeric fields
if main_record_set_id is not None:
    df_main = dataframes[main_record_set_id]
    print(f"Columns in {main_record_set_id}: {df_main.columns.tolist()}")
    # Try to automatically find a numeric field
    numeric_field_id = None
    for col in df_main.columns:
        if np.issubdtype(df_main[col].dropna().values[:10].dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # As fallback, cast all columns to numeric and pick first with >1 unique values
        for col in df_main.columns:
            try:
                cand = pd.to_numeric(df_main[col], errors='coerce')
                if cand.nunique() > 1:
                    numeric_field_id = col
                    break
            except Exception:
                continue
    if numeric_field_id is not None:
        print(f"Using numeric field (by @id): {numeric_field_id}")
        # Coerce numeric values safely
        df_main[numeric_field_id] = pd.to_numeric(df_main[numeric_field_id], errors='coerce')
        threshold = np.nanmean(df_main[numeric_field_id])
        filtered_df = df_main[df_main[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.3f}:")
        print(filtered_df.head())
        # Normalization
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Find a grouping/categorical field (assume first non-numeric field)
        group_field_id = None
        for col in df_main.columns:
            if (col != numeric_field_id) and (df_main[col].dtype == object):
                if df_main[col].nunique() > 1:
                    group_field_id = col
                    break
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("\nNo suitable group field found for grouping in this record set.")
    else:
        print("No numeric field detected in the selected record set.")
else:
    print("No data available for analysis.")


## 5. Visualization
Visualize the distribution of the normalized numeric field and grouped results. All references are by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id} (normalized)")
    plt.xlabel(f"{numeric_field_id}_normalized [@id={numeric_field_id}]")
    plt.show()

    if group_field_id is not None and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.barplot(
            x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(f"{group_field_id} [@id={group_field_id}]")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and process data from a FAIR² dataset using the `mlcroissant` Python library, referencing all core Croissant entities using their canonical `@id` identifiers. 

- **Metadata** loaded directly from the Croissant URL, exposing context, description, biases, and data use constraints.
- **Record Sets and Fields** explored and referenced by `@id`.
- **Data Extraction** performed for each record set, with further EDA on numeric and categorical fields (by `@id`).
- **EDA and Visualizations** of filtered and normalized numeric fields, grouped by categorical attributes.

Further analysis can extend from this template by customizing the field selections via `@id` or by integrating additional downstream ML workflows using the processed data.